In [1]:
import openai
from qdrant_client import QdrantClient


In [2]:
qdrant_client = QdrantClient(url ="http://localhost:6333")

In [3]:
##define embedding function
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding


In [5]:
##retreival function
def retrieve_data(query, k = 5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [7]:
retrieved_context = retrieve_data("Do you have a USB connectable fan for hot summers", k = 10)

In [8]:
retrieved_context

{'retrieved_context_ids': ['B0BRJS644Z',
  'B0BXC72RLD',
  'B099N9F3FP',
  'B0C8DBH7ZT',
  'B0CF57H28T',
  'B0BM9THPDQ',
  'B0C9QZS95R',
  'B0BYD7PGV1',
  'B09VDLH5M6',
  'B0CC4HBS85'],
 'retrieved_context': ['Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that supp

In [11]:
##format the retrieved documents
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'],context['retrieved_context'],context['retrieved_context_ratings']):
        formatted_context += f"ID: {id}, Rating: {rating}, description:{chunk}\n"
    return formatted_context

In [12]:
preprocessed_context = process_context(retrieved_context)

In [13]:
print(preprocessed_context)

ID: B0BRJS644Z, Rating: 4.7, description:Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】 This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed Control Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】 Rubber Feet on the fan raise it up enough off of the surface it is sitting 

In [14]:
##create promp template function
def built_prompt(preprocessed_context, question):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock
    You will be given a question and list of retrieved context.

    Instructions:
    - Answer the question based on the provided context only
    - Do not use markdown formatting
    - Never use word context and refer to it as the available products.

    Context: {preprocessed_context}
    Question: {question}
    
    """
    return prompt

In [17]:
prompt = built_prompt(preprocessed_context, "Do you have a USB connectable fan for hot summers ?")

In [18]:
print(prompt)


    You are a shopping assistant that can answer questions about the products in stock
    You will be given a question and list of retrieved context.

    Instructions:
    - Answer the question based on the provided context only
    - Do not use markdown formatting
    - Never use word context and refer to it as the available products.

    Context: ID: B0BRJS644Z, Rating: 4.7, description:Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】 This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed Control Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various enviro

In [19]:
##generate answer function
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "user", "content": prompt}
        ],
        reasoning_effort = "none"
    )
    return response.choices[0].message.content

In [21]:
print(generate_answer(prompt))

Yes. We have a few USB-connectable options:

1) Marame 120mm 5V USB Powered Fan with Speed Controller
- Works on a USB connection
- Includes an on-cord speed controller (off/low/medium/high)
- Designed to help prevent devices from overheating

2) HZD Desk Fan Rechargeable (USB-powered)
- Compact personal desk fan
- USB-powered via a USB cable (this one notes it does not include a battery)
- 3 speed settings (low/medium/high)
- Quiet operation (claimed under 50dB)

If you tell me whether you want a small personal fan for your desk or a larger 120mm fan for cooling equipment, I can help you choose.


In [23]:
##combine RAG pipeline components
def rag_pipeline( question,top_k = 5):
    
    retrieved_context = retrieve_data(question, k = top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = built_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer

In [24]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers ?"))

Yes. We have USB connectable fans that are suitable for hot summers:

1) Marame 120mm 5V USB Powered Fan with Speed Controller (ID: B0BRJS644Z)
- USB powered
- Speed controller with off/low/medium/high
- Designed for cooling devices and electronics

2) HZD Desk Fan Rechargeable, Mini Portable Fan (ID: B0BXC72RLD)
- USB powered (it states it does not include a battery)
- 3 speeds (low/medium/high)
- Quiet operation and includes a USB cable

If you tell me whether you need a larger cooling fan for electronics (router/TV box) or a small personal desk fan, I can recommend the best fit.
